## Reading Data from an API

A web API is an API over the web.

Think of an API like a restaurant's menu: you don't need to know how the kitchen works internally, you just pick an item from a fixed menu, and the kitchen sends back what you asked for. 

A web API works the same way: you send a request to a fixed endpoint, and it sends back data, usually in a clean, structured format like JSON.

In this lab, we'll pull real currency exchange rate data from a free public API and turn it into a pandas DataFrame we can explore.

### Setting up

We'll use Python's built-in `urllib` to make the request (no extra installation needed) and `json` to parse the response.

In [2]:
import urllib.request
import urllib.error
import json

### About the Frankfurter API

Frankfurter is a free, open-source currency exchange rate API, sourced from the European Central Bank (and other central banks). It's well-established, historical data goes back to **1948** and covers **201 currencies**.

**Endpoints:**
- **Latest rates:** `/v1/latest?base=USD`
- **Historical (specific date):** `/v1/1999-01-04?base=USD&symbols=EUR`
- **Time series (date range):** `/v1/2010-01-01..2010-01-31`
- **List of currencies:** `/v1/currencies`: returns currency codes along with their full names (e.g. `"EUR": "Euro"`)


### Making our request

Let's fetch the latest exchange rates, using the US Dollar as our base currency.

In [ ]:
url = "https://api.frankfurter.dev/v1/latest?base=USD"

#This is the address of the API endpoint we're calling. 
#?base=USD: This is a query parameter.

# Some APIs block requests that don't look like they're coming from a browser,
# so we set a User-Agent header

req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

try:
    response = urllib.request.urlopen(req) #urlopen(req) returns a response object, and .read() pulls the raw content out of it
    data = response.read()
    print("Request successful!")

except urllib.error.HTTPError as e:
    print(f"HTTP Error: {e.code} - {e.reason}") 
    
except urllib.error.URLError as e:
    print(f"URL Error: {e.reason}")

Request successful!


Two separate except blocks, catching two different failure types:

- HTTPError: The request reached the server just fine, but the server responded with an error (like 404 Not Found, or the 403 Forbidden).
- URLError: The request never even reached the server at all (e.g. no internet connection, or a typo in the URL/domain that doesn't resolve).

In [5]:
data

b'{"amount":1.0,"base":"USD","date":"2026-08-21","rates":{"AUD":1.3951,"BRL":5.1729,"CAD":1.374,"CHF":0.79947,"CNY":6.7206,"CZK":20.614,"DKK":6.3901,"EUR":0.85477,"GBP":0.73228,"HKD":7.8405,"HUF":310.09,"IDR":17659,"ILS":2.9846,"INR":95.7,"ISK":121.04,"JPY":158.7,"KRW":1384.23,"MXN":16.898,"MYR":4.0385,"NOK":9.2893,"NZD":1.6703,"PHP":61.66,"PLN":3.6822,"RON":4.4929,"SEK":9.4559,"SGD":1.2683,"THB":32.675,"TRY":48.066,"ZAR":16.0044}}'

data is just a string of characters that happens to look dictionary-shaped. `json.loads()` is the function that actually converts that text into a real Python object: a dictionary

In [6]:
rates_data = json.loads(data)
type(rates_data)

dict

`rates_data` is a dictionary with four keys: `amount`, `base`, `date`, and `rates`, where `rates` is itself a nested dictionary mapping currency codes to exchange rates.

In [13]:
rates_data

{'amount': 1.0,
 'base': 'USD',
 'date': '2026-08-21',
 'rates': {'AUD': 1.3951,
  'BRL': 5.1729,
  'CAD': 1.374,
  'CHF': 0.79947,
  'CNY': 6.7206,
  'CZK': 20.614,
  'DKK': 6.3901,
  'EUR': 0.85477,
  'GBP': 0.73228,
  'HKD': 7.8405,
  'HUF': 310.09,
  'IDR': 17659,
  'ILS': 2.9846,
  'INR': 95.7,
  'ISK': 121.04,
  'JPY': 158.7,
  'KRW': 1384.23,
  'MXN': 16.898,
  'MYR': 4.0385,
  'NOK': 9.2893,
  'NZD': 1.6703,
  'PHP': 61.66,
  'PLN': 3.6822,
  'RON': 4.4929,
  'SEK': 9.4559,
  'SGD': 1.2683,
  'THB': 32.675,
  'TRY': 48.066,
  'ZAR': 16.0044}}

In [14]:
print(rates_data.keys())
print(f"Base currency: {rates_data['base']}")
print(f"Date: {rates_data['date']}")

dict_keys(['amount', 'base', 'date', 'rates'])
Base currency: USD
Date: 2026-08-21


In [9]:
# The actual exchange rates are nested one level deeper
rates_data['rates']

{'AUD': 1.4072,
 'BRL': 5.1936,
 'CAD': 1.377,
 'CHF': 0.79899,
 'CNY': 6.7236,
 'CZK': 20.677,
 'DKK': 6.4,
 'EUR': 0.85609,
 'GBP': 0.73388,
 'HKD': 7.8438,
 'HUF': 312.56,
 'IDR': 17797,
 'ILS': 2.992,
 'INR': 95.71,
 'ISK': 121.56,
 'JPY': 158.76,
 'KRW': 1396.35,
 'MXN': 16.9906,
 'MYR': 4.044,
 'NOK': 9.3335,
 'NZD': 1.6828,
 'PHP': 61.728,
 'PLN': 3.6973,
 'RON': 4.4958,
 'SEK': 9.4919,
 'SGD': 1.2722,
 'THB': 32.915,
 'TRY': 47.953,
 'ZAR': 16.174}

### From dictionary to DataFrame

Now let's turn the rates dictionary into a proper pandas DataFrame, one row per currency, with its code and exchange rate.

In [13]:
import pandas as pd

rates_dict = rates_data['rates']
rates_df = pd.DataFrame(list(rates_dict.items()), columns=['Currency', 'Rate'])
rates_df.head()

,Currency,Rate
0,AUD,1.40720
1,BRL,5.19360
2,CAD,1.37700
3,CHF,0.79899
4,CNY,6.72360


In [17]:
rates_df.shape

(29, 2)